# Luminaria escolar con detección de movimiento

Este cuaderno reúne la lógica de detección de movimiento del proyecto de titulación y la corrección del temporizador.

## Detector y temporizador

El detector compara cada fotograma con un fondo de referencia. El temporizador se reinicia si el movimiento se interrumpe y confirma un solo evento por episodio cuando se cumple el tiempo configurado.

In [ ]:
import cv2

class MotionDetector:
    def __init__(self, minimum_area=500):
        self.minimum_area = minimum_area
        self.background = None

    def detect(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)
        if self.background is None:
            self.background = gray
            return []

        difference = cv2.absdiff(self.background, gray)
        _, mask = cv2.threshold(difference, 25, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, None, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = [
            cv2.boundingRect(contour)
            for contour in contours
            if cv2.contourArea(contour) >= self.minimum_area
        ]
        if not boxes:
            self.background = gray
        return boxes

class MotionEvent:
    def __init__(self, seconds):
        self.seconds = seconds
        self.started_at = None
        self.confirmed = False

    def update(self, moving, now):
        if not moving:
            self.started_at = None
            self.confirmed = False
            return False
        if self.started_at is None:
            self.started_at = now
        if not self.confirmed and now - self.started_at >= self.seconds:
            self.confirmed = True
            return True
        return False